# Ingeniería de variables y KPIs — RE/MAX CABA

Este notebook parte de la base limpia generada en `02_limpieza_remax.ipynb` y construye las variables necesarias para calcular los KPIs del proyecto: barrio y comuna oficiales, variables derivadas del texto de los avisos, estado de la propiedad, superficie homogeneizada, benchmark de comparables, gap de subvaluación y Precio Esperado Reciclado.

**Entrada:** `data/processed/remax_deptos_limpio.csv` (9.500 avisos).

**Fuentes externas:** límites oficiales de barrios de BA Data, que se descargan en `data/external/`.

**Dependencias:** `src/limpieza.py`, `src/patrones.py` y `src/diagnostico.py`.

In [1]:
# 1. ENTORNO E IMPORTS

import os

REPO_URL = "https://github.com/DTosoratti/tp-analitica-descriptiva-real-estate.git"
REPO_DIR = "/content/tp-analitica-descriptiva-real-estate"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

import pandas as pd
import numpy as np
from pathlib import Path
import geopandas as gpd
import requests

from src.limpieza import resumen_calidad, auditar_categorica, excluir, modificar, tabla_registro
from src.patrones import PATRONES

# Rutas centralizadas
PROCESSED_DIR = Path("data/processed")
EXTERNAL_DIR = Path("data/external")
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

BASE_LIMPIA = PROCESSED_DIR / "remax_deptos_limpio.csv"

Cloning into 'tp-analitica-descriptiva-real-estate'...
remote: Enumerating objects: 296, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 296 (delta 71), reused 15 (delta 15), pack-reused 159 (from 1)
Receiving objects: 100% (296/296), 40.48 MiB | 17.14 MiB/s, done.
Resolving deltas: 100% (134/134), done.
/content/tp-analitica-descriptiva-real-estate


In [2]:
# 2. CARGA DE LA BASE LIMPIA

df = pd.read_csv(BASE_LIMPIA)

print("Dimensiones:", df.shape)
print("IDs duplicados:", int(df["ID"].duplicated().sum()))
print("Avisos sin coordenadas:", int(df[["Latitud", "Longitud"]].isna().any(axis=1).sum()))

Dimensiones: (9500, 41)
IDs duplicados: 0
Avisos sin coordenadas: 0


### Renombre de la columna `Estado`

La base de la fuente incluye una columna llamada `Estado` que no describe la condición física del inmueble, sino el estado comercial de la publicación en RE/MAX: `active` (publicada), `reserved` (reservada) o `negotiation` (en negociación).

En este proyecto, en cambio, el estado de la propiedad (necesita obra, buen estado o reciclada) es una variable central del análisis, que se construye en este notebook como `Estado_Propiedad` a partir del título y la descripción. Para evitar que ambas variables se confundan, la columna de la fuente se renombra como `Estado_Publicacion`.

In [3]:
df = df.rename(columns={"Estado": "Estado_Publicacion"})

print("Valores de Estado_Publicacion:")
display(auditar_categorica(df, "Estado_Publicacion"))

Valores de Estado_Publicacion:


,Estado_Publicacion,Cantidad,Porcentaje
0,active,7164,75.41
1,reserved,1644,17.31
2,negotiation,692,7.28


**Resultado.** El 75,4% de los avisos está activo, el 17,3% reservado y el 7,3% en negociación. Una cuarta parte de la oferta capturada tiene, por lo tanto, una operación en curso. Esta variable aporta información sobre la liquidez: una reserva o negociación indica que hubo demanda al precio publicado. Se conserva para el análisis exploratorio, por ejemplo para comparar el gap de subvaluación de los avisos reservados con el de los activos.

## 3. Barrio y comuna oficiales

La fuente informa 74 etiquetas de ubicación, que incluyen subzonas y denominaciones informales (por ejemplo, "Palermo Hollywood", "Belgrano Chico" o "Barrio River") y no se corresponden con la división oficial de la Ciudad. Como el barrio es la variable principal para construir los grupos de propiedades comparables, cada aviso se asigna a su barrio y comuna oficiales mediante una unión espacial: se determina en qué polígono oficial cae el punto definido por su latitud y longitud.

**Fuente:** [BA Data — Barrios](https://data.buenosaires.gob.ar/dataset/barrios). Límites oficiales de los barrios de la Ciudad Autónoma de Buenos Aires, en formato GeoJSON. Licencia CC-BY-2.5-AR. Actualización trimestral; versión utilizada: última actualización del 29 de julio de 2026.

Primero se descarga el archivo y se revisa su estructura.

In [4]:
# 3.1 DESCARGA DE LOS POLÍGONOS DE BARRIOS

URL_BARRIOS = ("https://cdn.buenosaires.gob.ar/datosabiertos/datasets/"
               "innovacion-transformacion-digital/barrios/barrios.geojson")
RUTA_BARRIOS = EXTERNAL_DIR / "barrios_caba.geojson"

if not RUTA_BARRIOS.exists():
    respuesta = requests.get(URL_BARRIOS, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    respuesta.raise_for_status()
    RUTA_BARRIOS.write_bytes(respuesta.content)

barrios = gpd.read_file(RUTA_BARRIOS).to_crs("EPSG:4326")

print("Cantidad de barrios oficiales:", len(barrios))
print("Cantidad de comunas:", barrios["comuna"].nunique())
display(barrios[["nombre", "comuna"]].head())

Cantidad de barrios oficiales: 48
Cantidad de comunas: 15


,nombre,comuna
0,Agronomia,15
1,Almagro,5
2,Balvanera,3
3,Barracas,4
4,Belgrano,13


### 3.2 Unión espacial

Cada aviso se representa como un punto a partir de su longitud y latitud, y se identifica el polígono de barrio oficial que lo contiene (predicado *within*). Ambas capas usan el mismo sistema de coordenadas geográficas (WGS84). Los avisos sin coordenadas o con coordenadas fuera de los límites de la Ciudad quedan sin barrio oficial asignado y se analizan por separado.

Luego se compara la etiqueta de ubicación informada por RE/MAX con el barrio oficial asignado, para medir en qué medida la etiqueta de la fuente coincide con la división oficial.

In [5]:
# 3.2 UNIÓN ESPACIAL: AVISO → BARRIO Y COMUNA OFICIALES

from src.diagnostico import normalizar_etiqueta

puntos = gpd.GeoDataFrame(
    df[["ID"]],
    geometry=gpd.points_from_xy(df["Longitud"], df["Latitud"]),
    crs="EPSG:4326",
)

union = gpd.sjoin(puntos, barrios[["nombre", "comuna", "geometry"]],
                  how="left", predicate="within")

# Un punto sobre el límite exacto entre dos barrios podría coincidir con ambos: se toma el primero
union = union[~union.index.duplicated(keep="first")]

df["Barrio_Oficial"] = union["nombre"]
df["Comuna"] = union["comuna"]

sin_barrio = df["Barrio_Oficial"].isna()
print("Avisos con barrio oficial asignado:", int((~sin_barrio).sum()))
print("Avisos sin barrio oficial:", int(sin_barrio.sum()))

print("\nAvisos sin barrio oficial (coordenadas y etiqueta de la fuente):")
display(df.loc[sin_barrio, ["ID", "Titulo", "Barrio", "Latitud", "Longitud"]])

# Comparación entre la etiqueta de RE/MAX y el barrio oficial
coincide = (df["Barrio"].apply(normalizar_etiqueta)
            == df["Barrio_Oficial"].apply(normalizar_etiqueta))
print("\nEtiqueta de RE/MAX igual al barrio oficial:",
      f"{coincide[~sin_barrio].mean() * 100:.1f}% de los avisos asignados")

print("\nEtiquetas de RE/MAX que no coinciden con el barrio oficial (las 20 más frecuentes):")
display(
    df.loc[~coincide & ~sin_barrio]
    .groupby(["Barrio", "Barrio_Oficial"]).size()
    .sort_values(ascending=False).head(20)
    .rename("Avisos").to_frame()
)

Avisos con barrio oficial asignado: 9496
Avisos sin barrio oficial: 4

Avisos sin barrio oficial (coordenadas y etiqueta de la fuente):


,ID,Titulo,Barrio,Latitud,Longitud
357,656360,VENTA MONOAMBIENTE CON BALCON APTO PROFESIONAL,San Nicolás,-34.547814,-58.684187
8111,546854,DEPARTAMENTO VENTA CABALLITO GOYENA PISO 260M2,Caballito,-34.582786,-58.564961
8329,537073,VENTA DEPARTAMENTO 3 AMBIENTES EN CIUDADELA,Liniers,-34.640158,-58.530054
9282,383517,-Departamento 2 AMBIENTES EN FLORESTA,Floresta,-36.297418,-55.711670



Etiqueta de RE/MAX igual al barrio oficial: 83.8% de los avisos asignados

Etiquetas de RE/MAX que no coinciden con el barrio oficial (las 20 más frecuentes):


,,Avisos
Barrio,Barrio_Oficial,
Barrio Norte,Recoleta,151
Las Cañitas,Palermo,91
Palermo Hollywood,Palermo,71
Villa General Mitre,Villa Gral. Mitre,67
Caballito Sur,Caballito,63
Recoleta,Retiro,52
Palermo Soho,Palermo,47
Palermo Chico,Palermo,46
Belgrano R,Belgrano,45


### Resultado de la unión espacial

El 99,96% de los avisos (9.496) quedó asignado a un barrio oficial. Los cuatro avisos sin asignar tienen coordenadas fuera de los límites de la Ciudad:

- Uno corresponde efectivamente a una propiedad fuera del alcance: el título indica que está en Ciudadela, en la provincia de Buenos Aires. Se excluye y se registra en la tabla de excepciones manuales.
- Los otros tres son propiedades de CABA con coordenadas mal cargadas (una de ellas ubicada en el mar). Se les asigna el barrio a partir de la etiqueta de la fuente, que en los tres casos corresponde a un barrio oficial. Además, se marcan con `Flag_Coordenadas_Invalidas` y se eliminan sus coordenadas, para que no intervengan en análisis espaciales.

La etiqueta de ubicación de RE/MAX coincide con el barrio oficial en el 83,8% de los avisos. Las diferencias son de tres tipos:

1. **Subzonas informales** que se corresponden con su barrio oficial, como Barrio Norte (Recoleta), Las Cañitas, Palermo Hollywood, Palermo Soho y Palermo Chico (Palermo), o Belgrano R (Belgrano). Sin la unificación, un mismo barrio quedaría fragmentado en varios grupos de comparables.
2. **Diferencias de escritura**, como "Villa General Mitre" y "Villa Gral. Mitre".
3. **Asignación a un barrio vecino**, por ejemplo avisos etiquetados como Recoleta ubicados en Retiro, o como Belgrano ubicados en Colegiales. En varios de estos casos la etiqueta corresponde al barrio de mayor valor de la zona, lo que sugiere un posible **sesgo comercial de la fuente**: la ubicación declarada tiende a asociarse con el barrio más valorado. Usar la etiqueta original habría comparado estas propiedades con un segmento de precio que no les corresponde.

Por estas razones, en todo el análisis se utiliza `Barrio_Oficial`, y la etiqueta original se conserva solo como referencia.

In [6]:
# 3.3 TRATAMIENTO DE LOS AVISOS SIN BARRIO OFICIAL

from src.limpieza import cargar_excepciones, aplicar_excepciones

registro = []

# Exclusión documentada en la tabla de excepciones (propiedad fuera de CABA)
excepciones = cargar_excepciones("data/excepciones_manuales.csv")
df, reporte = aplicar_excepciones(df, excepciones, registro, etapa="features")
display(reporte)

# Coordenadas inválidas: barrio asignado por la etiqueta de la fuente
mapa_barrios = {normalizar_etiqueta(n): n for n in barrios["nombre"]}
mapa_comunas = dict(zip(barrios["nombre"], barrios["comuna"]))

sin_barrio = df["Barrio_Oficial"].isna()

df["Flag_Coordenadas_Invalidas"] = False
df = modificar(
    df, sin_barrio, "Flag_Coordenadas_Invalidas", True,
    paso="Barrio oficial: coordenadas inválidas",
    criterio="Coordenadas fuera de los límites de CABA en una propiedad de la Ciudad",
    registro=registro,
)

df.loc[sin_barrio, "Barrio_Oficial"] = (
    df.loc[sin_barrio, "Barrio"].apply(normalizar_etiqueta).map(mapa_barrios)
)
df.loc[sin_barrio, "Comuna"] = df.loc[sin_barrio, "Barrio_Oficial"].map(mapa_comunas)

for col in ["Latitud", "Longitud"]:
    df = modificar(
        df, sin_barrio, col, np.nan,
        paso="Barrio oficial: coordenadas inválidas",
        criterio="Se eliminan las coordenadas erróneas para excluirlas de los análisis espaciales",
        registro=registro,
    )

df["Comuna"] = df["Comuna"].astype("Int64")

# Renumerar el índice después de las exclusiones, para que las uniones (merge) posteriores no desalineen las filas
df = df.reset_index(drop=True)

print("Avisos con coordenadas inválidas:")
display(df.loc[df["Flag_Coordenadas_Invalidas"],
               ["ID", "Titulo", "Barrio", "Barrio_Oficial", "Comuna"]])
print("Sin barrio oficial:", int(df["Barrio_Oficial"].isna().sum()))
print("Sin comuna:", int(df["Comuna"].isna().sum()))
print("Dimensiones:", df.shape)

display(tabla_registro(registro))

,ID,accion,columna,valor_nuevo,motivo,encontrado
20,537073,excluir,NaN,NaN,Propiedad ubicada en Ciudadela (provincia de B...,True


Avisos con coordenadas inválidas:


,ID,Titulo,Barrio,Barrio_Oficial,Comuna
357,656360,VENTA MONOAMBIENTE CON BALCON APTO PROFESIONAL,San Nicolás,San Nicolas,1
8111,546854,DEPARTAMENTO VENTA CABALLITO GOYENA PISO 260M2,Caballito,Caballito,6
9281,383517,-Departamento 2 AMBIENTES EN FLORESTA,Floresta,Floresta,10


Sin barrio oficial: 0
Sin comuna: 0
Dimensiones: (9499, 44)


,Paso,Tipo,Criterio,Registros_afectados,Filas_antes,Filas_despues
0,Excepciones manuales (features),Exclusión,Exclusiones documentadas en excepciones_manual...,1,9500,9499
1,Barrio oficial: coordenadas inválidas,Modificación (Flag_Coordenadas_Invalidas),Coordenadas fuera de los límites de CABA en un...,3,9499,9499
2,Barrio oficial: coordenadas inválidas,Modificación (Latitud),Se eliminan las coordenadas erróneas para excl...,3,9499,9499
3,Barrio oficial: coordenadas inválidas,Modificación (Longitud),Se eliminan las coordenadas erróneas para excl...,3,9499,9499


**Resultado.** El aviso ubicado en Ciudadela se excluyó a partir de la tabla de excepciones manuales, y la base queda en 9.499 avisos. Los tres avisos de CABA con coordenadas erróneas recibieron el barrio y la comuna correspondientes a su etiqueta (San Nicolás, Caballito y Floresta), quedaron marcados con `Flag_Coordenadas_Invalidas` y se eliminaron sus coordenadas. Todos los avisos de la base tienen barrio y comuna oficiales asignados.

## 4. Variables derivadas del texto

El título y la descripción de los avisos contienen información que no figura en las columnas estructuradas de la fuente, como la luminosidad, la vista o la necesidad de refacción. Estas características se extraen mediante expresiones regulares y se representan como variables booleanas.

Como el objetivo es detectar la presencia de características concretas, el texto se normaliza en forma moderada: se pasa a minúsculas, se eliminan los acentos y se unifican los espacios. No se eliminan palabras vacías (*stop-words*), porque las negaciones invierten el sentido ("no requiere refacción"). En su lugar, para las características que suelen aparecer negadas, se descarta toda coincidencia precedida por "no", "sin" o "ni" dentro de las tres palabras anteriores.

Todos los patrones están definidos en `src/patrones.py`.

In [7]:
# 4.1 TEXTO UNIFICADO Y NORMALIZADO

from src.limpieza import normalizar_texto, contiene_sin_negacion, marcar_amenities
from src.patrones import TEXTO_BOOLEANAS, BOOLEANAS_CON_NEGACION, NEGACION, AMENITIES

df["Texto_Analisis"] = df["Titulo"].fillna("") + " " + df["Descripcion"].fillna("")
df["Texto_Normalizado"] = normalizar_texto(df["Texto_Analisis"])

print("Avisos:", len(df))
print("Avisos sin descripción (solo título):", int(df["Descripcion"].isna().sum()))
print("Largo del texto normalizado (caracteres):")
display(df["Texto_Normalizado"].str.len().describe().round(0).to_frame("Caracteres"))

Avisos: 9499
Avisos sin descripción (solo título): 54
Largo del texto normalizado (caracteres):


,Caracteres
count,9499.0
mean,1623.0
std,861.0
min,3.0
25%,1090.0
50%,1487.0
75%,1955.0
max,16713.0


**Resultado.** Se construyó un texto unificado para los 9.499 avisos. Los 54 avisos sin descripción (los de la falla del endpoint de detalle) se analizan solo a partir del título, por lo que tienen menos probabilidad de mencionar cualquier característica. La mitad de los textos supera los 1.487 caracteres, lo que da una base amplia para la extracción de variables.

### 4.2 Variables booleanas descriptivas

Se construyen ocho variables a partir del texto: `Luminoso`, `Vista`, `Silencioso`, `Contrafrente`, `Al_Frente`, `Apto_Profesional`, `Reciclado_Texto` y `A_Refaccionar`. En las que suelen aparecer negadas (luminoso, silencioso, reciclado y a refaccionar) se aplica el control de negación.

Estas variables indican la **mención explícita** de una característica. Un valor `False` significa que el aviso no la menciona, no que la propiedad no la tenga. Por eso se interpretan como señales de lo que el anunciante decide destacar.

In [8]:
# 4.2 VARIABLES BOOLEANAS DESDE EL TEXTO

resumen = []
for variable, patron in TEXTO_BOOLEANAS.items():
    menciona = df["Texto_Normalizado"].str.contains(patron, regex=True)
    if variable in BOOLEANAS_CON_NEGACION:
        df[variable] = contiene_sin_negacion(df["Texto_Normalizado"], patron, NEGACION)
    else:
        df[variable] = menciona
    resumen.append({
        "Variable": variable,
        "Avisos": int(df[variable].sum()),
        "Porcentaje": round(df[variable].mean() * 100, 2),
        "Descartados por negación": int((menciona & ~df[variable]).sum()),
    })

print("Variables booleanas extraídas del texto:")
display(pd.DataFrame(resumen).set_index("Variable"))

Variables booleanas extraídas del texto:


,Avisos,Porcentaje,Descartados por negación
Variable,,,
Luminoso,4939,51.99,1
Vista,2003,21.09,0
Silencioso,497,5.23,0
Contrafrente,1939,20.41,0
Al_Frente,2884,30.36,0
Apto_Profesional,1159,12.20,0
Reciclado_Texto,1087,11.44,0
A_Refaccionar,318,3.35,2


**Resultado.** Las características más mencionadas son la luminosidad (52% de los avisos), la ubicación al frente (30%), la vista (21%) y el contrafrente (20%). Las menciones de reciclado aparecen en el 11,4% de los avisos y las de necesidad de refacción, en el 3,4%. El control de negación descartó solo tres coincidencias, lo que indica que en estos avisos las características se mencionan casi siempre en forma afirmativa; aun así, el control evita errores relevantes como clasificar "no necesita refacción" como una unidad a refaccionar.

In [9]:
# 4.3 AUDITORÍA DE LOS PATRONES

import re
def fragmento(texto, patron, ancho=60):
    """Texto alrededor de la primera coincidencia del patrón."""
    m = re.search(patron, texto)
    return texto[max(0, m.start() - ancho):m.end() + ancho] if m else ""

for variable, patron in TEXTO_BOOLEANAS.items():
    casos = df.loc[df[variable], "Texto_Normalizado"]
    print(f"\n- {variable} ({len(casos)} avisos) — muestra -")
    for texto in casos.sample(min(5, len(casos)), random_state=42):
        print("  ...", fragmento(texto, patron), "...")

# Casos descartados por negación (para verificar que el control funciona)
for variable in ["A_Refaccionar", "Reciclado_Texto"]:
    patron = TEXTO_BOOLEANAS[variable]
    descartados = df.loc[
        df["Texto_Normalizado"].str.contains(patron, regex=True) & ~df[variable],
        "Texto_Normalizado",
    ]
    print(f"\n- {variable}: descartados por negación ({len(descartados)}) -")
    for texto in descartados.head(5):
        print("  ...", fragmento(texto, patron), "...")


- Luminoso (4939 avisos) — muestra -
  ... ertos y 5m2 de balcon. living comedor de generosas medidas, muy luminoso y con salida al balcon con vista panoramica, abierta y con  ...
  ... ral ventilacion cruzada edificio con 5 ascensores ambientes luminosos y funcionales potencial de revalorizacion inversion recomen ...
  ... venta depto 2amb en nunez 2 ambientes luminoso | frente a estacion nunez excelente oportunidad en una de l ...
  ... ra vivienda como para inversion. al ingresar, nos recibe un luminoso living-comedor con salida a un balcon corrido, un espacio t ...
  ...  sin necesidad de realizar reformas. cuenta con un amplio y luminoso living-comedor, una comoda habitacion, cocina semiintegrada ...

- Vista (2003 avisos) — muestra -
  ... arrilla y cocina equipada. terraza de uso comun y solarium. vista panoramica. frente a la estacion jose cubas del metrobus, lo que facil ...
  ... cede, a traves de un gran ventanal, al balcon al frente con vista abierta, un espacio ideal para

**Auditoría de los patrones.** La revisión de una muestra de coincidencias por variable permitió corregir dos problemas antes de usar las variables:

- La ventana de negación abarcaba frases distintas: en "sin expensas, recientemente refaccionada", la palabra "sin" anulaba la mención de refacción. Se limitó la ventana para que se corte en los signos de puntuación.
- El patrón de ubicación al frente incluía expresiones como "al frente del edificio", que no describen la orientación de la unidad. Se excluyeron las construcciones "al frente de".

La muestra también muestra limitaciones que se mantienen y deben considerarse al interpretar las variables:

- `Reciclado_Texto` incluye menciones de reciclado de un solo ambiente (por ejemplo, "baño refaccionado a nuevo"), por lo que no equivale a una unidad reciclada en forma integral. Esa distinción se hace en la clasificación del estado (sección 5).
- `Silencioso` incluye descripciones del entorno ("barrio muy tranquilo") además de la unidad.
- `Al_Frente` y `Contrafrente` no son excluyentes: un aviso puede tener un balcón al frente y un patio al contrafrente.

### 4.4 Amenities y cochera

Las amenities se identifican combinando dos fuentes: la lista estructurada que publica RE/MAX (columna `Amenities`) y las menciones en el texto del aviso. La lista es la fuente más confiable, pero falta en el 17% de los avisos; el texto permite recuperar parte de esa información. Una amenity se considera presente si aparece en cualquiera de las dos fuentes.

Para la cochera, la sola mención no alcanza: algunos avisos aclaran que la cochera es opcional, no está incluida en el precio o se vende por separado. Por eso se distingue entre `Cochera_Mencion` (el aviso menciona una cochera) y `Cochera_Incluida` (la menciona y no aclara que esté excluida del precio).

In [10]:
# 4.4 AMENITIES Y COCHERA

amen = marcar_amenities(df, AMENITIES)
df[list(AMENITIES)] = amen[list(AMENITIES)]

resumen_amen = pd.DataFrame({
    "Por lista": amen[[f"{a}_Lista" for a in AMENITIES]].sum().values,
    "Por texto": amen[[f"{a}_Texto" for a in AMENITIES]].sum().values,
    "Total": df[list(AMENITIES)].sum().values,
}, index=list(AMENITIES))
resumen_amen["Solo por texto"] = (
    resumen_amen["Total"] - resumen_amen["Por lista"]
)
resumen_amen["Porcentaje"] = (resumen_amen["Total"] / len(df) * 100).round(2)

print("Amenities (lista estructurada + texto):")
display(resumen_amen.sort_values("Total", ascending=False))

# Cochera
df["Cochera_Mencion"] = df["Texto_Normalizado"].str.contains(PATRONES["cochera"], regex=True)
df["Cochera_No_Incluida"] = df["Texto_Normalizado"].str.contains(PATRONES["cochera_no_incluida"], regex=True)
df["Cochera_Incluida"] = df["Cochera_Mencion"] & ~df["Cochera_No_Incluida"]

print("Mencionan cochera:", int(df["Cochera_Mencion"].sum()))
print("   de las cuales aclaran que no está incluida:", int((df["Cochera_Mencion"] & df["Cochera_No_Incluida"]).sum()))
print("Cochera incluida:", int(df["Cochera_Incluida"].sum()),
      f"({df['Cochera_Incluida'].mean() * 100:.1f}%)")

Amenities (lista estructurada + texto):


,Por lista,Por texto,Total,Solo por texto,Porcentaje
Balcon,4829,6182,6493,1664,68.35
Ascensor,3954,2707,4962,1008,52.24
Terraza,1314,2022,2440,1126,25.69
Parrilla,1110,1867,2038,928,21.45
Baulera,943,1617,1755,812,18.48
SUM,1053,1430,1593,540,16.77
Pileta,1216,1478,1572,356,16.55
Patio,931,1196,1351,420,14.22
Solarium,850,953,1215,365,12.79
Seguridad,990,627,1205,215,12.69


Mencionan cochera: 2724
   de las cuales aclaran que no está incluida: 199
Cochera incluida: 2525 (26.6%)


**Resultado.** La combinación de la lista estructurada con el texto aumenta la cobertura de todas las amenities. Por ejemplo, el balcón figura en la lista en 4.829 avisos, y el texto permite identificarlo en otros 1.664. El texto aporta especialmente en las características de la propia unidad (balcón, terraza, baulera), que la lista de la fuente informa de forma incompleta.

Hay que tener en cuenta una limitación: el texto no siempre distingue entre una característica propia y una del edificio. Por ejemplo, "terraza" puede referirse a una terraza propia o a una de uso común, y "parrilla" puede estar en la unidad o en el edificio. Por eso estas variables indican la **disponibilidad** de la característica, ya sea propia o compartida.

La cochera se menciona en 2.724 avisos, de los cuales 199 aclaran que no está incluida en el precio (es opcional o se vende por separado). La variable `Cochera_Incluida` identifica 2.525 avisos (26,6%). Si se tomara solo la mención, se sobreestimaría la presencia de cochera y se compararía el precio de unidades sin cochera con el de unidades con cochera.

## 5. Estado de la propiedad

El estado de conservación se clasifica a partir del texto de cada aviso, en cuatro categorías: `Necesita_obra`, `Reciclada_refaccionada`, `Buen_estado` y `Sin_clasificar`. Como se explica en el README, es una **señal imperfecta**: la mayoría de los avisos no menciona la condición del inmueble, la mención depende de lo que el anunciante decide destacar y algunas menciones se refieren a un solo ambiente.

La clasificación sigue una jerarquía de señales. Cuando un aviso presenta señales contradictorias, gana la primera que se cumple:

1. **Necesita obra (señal fuerte):** "a refaccionar", "a reciclar", "para reciclar".
2. **Reciclada (integral):** menciones de reciclado de la unidad completa. No cuentan las que se refieren solo a un ambiente, como "baño reciclado", que se marcan como `Reciclado_Parcial`.
3. **Buen estado:** "excelente estado", "muy buen estado", "buen estado".
4. **Necesita obra (señal débil):** "a actualizar", "para modernizar", "requiere refacción".
5. **Reciclada (señal débil):** "completamente remodelado", "hecho a nuevo".

Todas las señales se aplican con control de negación. Después se aplican las correcciones manuales registradas en la tabla de excepciones (etapa `estado`). Finalmente se construye `Estado_Agrupado`, con tres niveles (`Necesita_obra`, `Buen_o_reciclada` y `Sin_clasificar`), que se usa para armar los comparables. `Sin_clasificar` se mantiene como categoría propia: no se imputa ni se asume buen estado.

In [11]:
# 5.1 SEÑALES DE ESTADO

from src.patrones import RECICLADO_INTEGRAL, RECICLADO_PARCIAL

texto = df["Texto_Normalizado"]

def senal(patron):
    return contiene_sin_negacion(texto, patron, NEGACION)

senales = pd.DataFrame({
    "Necesita_obra_fuerte": senal(PATRONES["necesita_obra_fuerte"]),
    "Reciclada": senal(PATRONES["reciclada"]) | senal(RECICLADO_INTEGRAL),
    "Reciclado_integral": senal(RECICLADO_INTEGRAL),
    "Mencion_parcial": texto.str.contains(RECICLADO_PARCIAL, regex=True),
    "Buen_estado": senal(PATRONES["buen_estado"]),
    "Necesita_obra_debil": senal(PATRONES["necesita_obra_debil"]),
    "Reciclada_debil": senal(PATRONES["reciclada_debil"]),
})

# Reciclado parcial: menciona un ambiente reciclado y no hay evidencia de reciclado integral
df["Reciclado_Parcial"] = senales["Mencion_parcial"] & ~senales["Reciclado_integral"]

print("Avisos con cada señal (un aviso puede tener varias):")
display(senales.sum().rename("Avisos").to_frame())
print("Reciclado parcial (sin evidencia de reciclado integral):", int(df["Reciclado_Parcial"].sum()))

Avisos con cada señal (un aviso puede tener varias):


,Avisos
Necesita_obra_fuerte,316
Reciclada,1258
Reciclado_integral,864
Mencion_parcial,696
Buen_estado,2371
Necesita_obra_debil,266
Reciclada_debil,20


Reciclado parcial (sin evidencia de reciclado integral): 343


In [12]:
# 5.2 CLASIFICACIÓN DEL ESTADO

reciclada_integral = senales["Reciclada"] & ~df["Reciclado_Parcial"]

condiciones = [
    senales["Necesita_obra_fuerte"],
    reciclada_integral,
    senales["Buen_estado"],
    senales["Necesita_obra_debil"],
    senales["Reciclada_debil"] & ~df["Reciclado_Parcial"],
]
categorias = ["Necesita_obra", "Reciclada_refaccionada", "Buen_estado",
              "Necesita_obra", "Reciclada_refaccionada"]

df["Estado_Propiedad"] = np.select(condiciones, categorias, default="Sin_clasificar")

# Correcciones manuales documentadas (etapa "estado")
df, reporte_estado = aplicar_excepciones(df, excepciones, registro, etapa="estado")
print("Excepciones manuales de estado:")
display(reporte_estado)

df["Estado_Agrupado"] = df["Estado_Propiedad"].replace({
    "Buen_estado": "Buen_o_reciclada",
    "Reciclada_refaccionada": "Buen_o_reciclada",
})

print("\nEstado de la propiedad:")
display(auditar_categorica(df, "Estado_Propiedad"))
print("Estado agrupado:")
display(auditar_categorica(df, "Estado_Agrupado"))

Excepciones manuales de estado:


,ID,accion,columna,valor_nuevo,motivo,encontrado
12,629386,reemplazar,Estado_Propiedad,Reciclada_refaccionada,Lectura manual de la descripción,True
13,627609,reemplazar,Estado_Propiedad,Reciclada_refaccionada,Lectura manual de la descripción,True
14,611305,reemplazar,Estado_Propiedad,Reciclada_refaccionada,Lectura manual de la descripción,True
15,610234,reemplazar,Estado_Propiedad,Reciclada_refaccionada,Lectura manual de la descripción,True
16,607641,reemplazar,Estado_Propiedad,Reciclada_refaccionada,Lectura manual de la descripción,True
17,607842,reemplazar,Estado_Propiedad,Necesita_obra,Lectura manual de la descripción,True
18,606775,reemplazar,Estado_Propiedad,Necesita_obra,Lectura manual de la descripción,True
19,573714,reemplazar,Estado_Propiedad,Necesita_obra,Lectura manual de la descripción,True



Estado de la propiedad:


,Estado_Propiedad,Cantidad,Porcentaje
0,Sin_clasificar,6034,63.52
1,Buen_estado,2049,21.57
2,Reciclada_refaccionada,977,10.29
3,Necesita_obra,439,4.62


Estado agrupado:


,Estado_Agrupado,Cantidad,Porcentaje
0,Sin_clasificar,6034,63.52
1,Buen_o_reciclada,3026,31.86
2,Necesita_obra,439,4.62


**Resultado.** El 63,5% de los avisos no menciona el estado del inmueble y queda como `Sin_clasificar`. Entre los clasificados, el 21,6% declara buen estado, el 10,3% está reciclado o refaccionado en forma integral y el 4,6% (439 avisos) necesita obra. Además, 343 avisos mencionan el reciclado de un solo ambiente sin evidencia de reciclado integral; se marcan como `Reciclado_Parcial` y no se cuentan como unidades recicladas.

Se aplicaron 8 de las 9 correcciones manuales de estado registradas en la tabla de excepciones. La restante corresponde a un aviso que no forma parte de la base analítica.

`Estado_Agrupado` reúne el 31,9% de avisos en buen estado o reciclados, que funcionan como referencia para el Precio Esperado Reciclado, y el 4,6% que necesita obra, que constituye el universo de análisis del flipping. La baja cobertura de la variable es una limitación: parte de los avisos sin clasificar probablemente necesitan obra y no lo declaran. La precisión de la clasificación se evaluará con la muestra de validación manual.

In [13]:
# 5.3 CONTROL FINAL DEL ESTADO

print("Total de avisos:", len(df))
print("Estado_Propiedad nulos:", df["Estado_Propiedad"].isna().sum())
print("Estado_Agrupado nulos:", df["Estado_Agrupado"].isna().sum())

print(
    "Suma de categorías Estado_Propiedad:",
    df["Estado_Propiedad"].value_counts().sum()
)

print(
    "Suma de categorías Estado_Agrupado:",
    df["Estado_Agrupado"].value_counts().sum()
)

Total de avisos: 9499
Estado_Propiedad nulos: 0
Estado_Agrupado nulos: 0
Suma de categorías Estado_Propiedad: 9499
Suma de categorías Estado_Agrupado: 9499


## 6. Superficie homogeneizada y grupos de comparables

Para comparar propiedades con distinta composición de superficie se calcula una **superficie homogeneizada**, que pondera la superficie cubierta al 100% y la descubierta (balcones, patios, terrazas) con un coeficiente menor, porque esa superficie vale menos. Se adopta un coeficiente de 0,5 como supuesto inicial, cuya sensibilidad se evaluará entre 0,3 y 0,7. Un único aviso no informa la superficie cubierta: para ese caso se usa la superficie total y se marca con `Flag_Superficie_Homogeneizada_Imputada`.

A partir de esa superficie se calcula el precio por m² homogeneizado, que es la unidad de comparación del benchmark.

In [14]:
# 6.1 SUPERFICIE HOMOGENEIZADA Y USD/M² HOMOGENEIZADO

COEF_DESCUBIERTA = 0.5   # supuesto inicial; sensibilidad entre 0,3 y 0,7

df["Superficie_Descubierta_m2"] = df["Superficie_Total_m2"] - df["Superficie_Cubierta_m2"]

# Si falta la superficie cubierta, se usa la total y se marca el caso
sin_cubierta = df["Superficie_Cubierta_m2"].isna()
df["Flag_Superficie_Homogeneizada_Imputada"] = sin_cubierta
df["Superficie_Homogeneizada_m2"] = np.where(
    sin_cubierta,
    df["Superficie_Total_m2"],
    df["Superficie_Cubierta_m2"] + COEF_DESCUBIERTA * df["Superficie_Descubierta_m2"],
)
df["USD_m2_Homogeneizado"] = df["Precio_Valor"] / df["Superficie_Homogeneizada_m2"]

print("Superficie descubierta negativa:", int((df["Superficie_Descubierta_m2"] < 0).sum()))
print("Superficie homogeneizada imputada con la total:", int(sin_cubierta.sum()))
print("USD/m² homogeneizado nulo o infinito:",
      int((~np.isfinite(df["USD_m2_Homogeneizado"])).sum()))

display(
    df[["Superficie_Total_m2", "Superficie_Homogeneizada_m2",
        "Precio_m2_Total", "USD_m2_Homogeneizado"]]
    .describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(1)
)

Superficie descubierta negativa: 0
Superficie homogeneizada imputada con la total: 1
USD/m² homogeneizado nulo o infinito: 0


,Superficie_Total_m2,Superficie_Homogeneizada_m2,Precio_m2_Total,USD_m2_Homogeneizado
count,9499.0,9499.0,9499.0,9499.0
mean,73.4,68.7,2180.3,2307.5
std,49.2,44.3,825.2,885.1
min,14.0,14.0,292.9,297.4
1%,22.0,22.0,782.9,847.3
25%,42.4,40.8,1637.0,1722.1
50%,60.0,56.9,2064.5,2183.1
75%,86.2,81.0,2563.5,2711.1
99%,265.1,239.5,4954.0,5253.6
max,703.0,608.0,15403.4,16755.3


**Resultado.** La superficie homogeneizada tiene una mediana de 56,9 m², frente a 60 m² de superficie total, porque la superficie descubierta se pondera al 50%. En consecuencia, el precio por m² homogeneizado (mediana de USD 2.183) es más alto que el calculado sobre la superficie total (USD 2.065), y refleja mejor el valor de las unidades con balcones, patios o terrazas amplias. Un único aviso sin superficie cubierta se calculó con la superficie total y quedó marcado.

### 6.2 Construcción de bandas para comparables

Para construir grupos de propiedades comparables se discretizan la superficie homogeneizada y la antigüedad.

La superficie se divide en terciles dentro de cada cantidad de ambientes, generando tres categorías relativas: `Chica`, `Media` y `Grande`. De esta forma, el tamaño de una propiedad se evalúa en relación con otras unidades de igual cantidad de ambientes.

La antigüedad se agrupa en las categorías `Hasta_10`, `11_30`, `31_50`, `Mas_50` y `Sin_dato`.

Estas variables se utilizarán posteriormente en la construcción jerárquica del benchmark y del Precio Esperado Reciclado (PER).

In [15]:
# 6.2 BANDAS PARA LOS GRUPOS DE COMPARABLES

from src.benchmark import grupo_ambientes, terciles_por_grupo, banda_antiguedad

df["Ambientes_Grupo"] = grupo_ambientes(df["Ambientes"])
df["Banda_Superficie"] = terciles_por_grupo(df["Superficie_Homogeneizada_m2"], df["Ambientes_Grupo"])
df["Banda_Antiguedad"] = banda_antiguedad(df["Antiguedad"])

print("Avisos por grupo de ambientes y banda de superficie:")
display(pd.crosstab(df["Ambientes_Grupo"], df["Banda_Superficie"], margins=True))

print("Límites de superficie homogeneizada (m²) de cada banda:")
display(
    df.groupby(["Ambientes_Grupo", "Banda_Superficie"])["Superficie_Homogeneizada_m2"]
    .agg(["min", "max"]).round(1).unstack()
)

print("Bandas de antigüedad:")
display(auditar_categorica(df, "Banda_Antiguedad"))

Avisos por grupo de ambientes y banda de superficie:


Banda_Superficie,Chica,Grande,Media,All
Ambientes_Grupo,,,,
1,509,509,509,1527
2,924,924,924,2772
3,1007,1007,1006,3020
4,575,576,575,1726
5+,151,152,151,454
All,3166,3168,3165,9499


Límites de superficie homogeneizada (m²) de cada banda:


min                  max              
Banda_Superficie Chica Grande  Media  Chica Grande  Media
Ambientes_Grupo                                          
1                 14.0   35.5   30.0   30.0  243.0   35.5
2                 21.2   48.0   39.9   39.9  134.5   47.9
3                 34.0   72.5   59.0   59.0  282.0   72.5
4                 40.3  112.0   85.5   85.5  376.0  112.0
5+                77.8  198.6  145.0  144.6  608.0  198.0

Bandas de antigüedad:


,Banda_Antiguedad,Cantidad,Porcentaje
0,Mas_50,3224,33.94
1,31_50,2663,28.03
2,Hasta_10,1682,17.71
3,11_30,1608,16.93
4,Sin_dato,322,3.39


**Resultado.** Los terciles de superficie dividen cada grupo de ambientes en tres partes iguales, con límites propios para cada tamaño de unidad. Por ejemplo, entre los departamentos de dos ambientes, la banda chica llega hasta los 40 m², la media va de 40 a 48 m² y la grande supera los 48 m². Cuando dos unidades tienen la misma superficie justo en el límite, el desempate se resuelve por orden de aparición, para mantener los tres grupos del mismo tamaño.

El stock es mayormente antiguo: el 62% de los departamentos tiene más de 30 años, y el 34%, más de 50. Solo el 3,4% no informa la antigüedad; esos avisos forman una banda propia, en lugar de imputarse.

### 6.3 Universo de referencia para comparables

Antes de construir el benchmark se define el conjunto de propiedades habilitadas para actuar como comparables.

Se excluyen las propiedades clasificadas como `Necesita_obra`, ya que su menor valor puede estar explicado por su condición física, así como los avisos identificados como extremos o con cantidad de ambientes dudosa.

Estas exclusiones afectan al conjunto utilizado como referencia, pero las propiedades se conservan en la base analítica para su posterior evaluación.

Adicionalmente, al calcular el benchmark de una propiedad, la propia observación evaluada no se considera entre sus comparables.

In [16]:
# 6.3 UNIVERSO DE REFERENCIA PARA EL BENCHMARK

exclusiones = pd.DataFrame({
    "Necesita obra": df["Estado_Propiedad"].eq("Necesita_obra"),
    "Valor extremo": df["Flag_Extremo"].fillna(False).astype(bool),
    "Ambientes dudosos": df["Flag_Ambientes_Dudoso"].fillna(False).astype(bool),
})
df["Flag_Comparable_Valido"] = ~exclusiones.any(axis=1)

print("Avisos excluidos como comparables, por criterio (un aviso puede cumplir varios):")
display(exclusiones.sum().rename("Avisos").to_frame())
print(f"Comparables válidos: {df['Flag_Comparable_Valido'].sum()} "
      f"({df['Flag_Comparable_Valido'].mean() * 100:.1f}%)")

Avisos excluidos como comparables, por criterio (un aviso puede cumplir varios):


,Avisos
Necesita obra,439
Valor extremo,3
Ambientes dudosos,264


Comparables válidos: 8815 (92.8%)


**Resultado.** 8.815 avisos (92,8%) pueden actuar como comparables. Se excluyen las 439 unidades que necesitan obra (para que el benchmark represente unidades en condiciones de habitabilidad), los 3 valores extremos y los 264 avisos con cantidad de ambientes dudosa, que podrían quedar asignados a un grupo que no les corresponde. Las propiedades excluidas como referencia se conservan en la base y reciben igualmente su benchmark.

## 7. Benchmark de propiedades comparables

Para cada propiedad se estima un benchmark de USD/m² homogeneizado utilizando únicamente propiedades habilitadas como comparables.

Se aplica una jerarquía de cuatro niveles:

1. Barrio × ambientes × banda de superficie × banda de antigüedad.
2. Barrio × ambientes × banda de superficie.
3. Comuna × ambientes × banda de superficie.
4. Comuna × ambientes.

Cada nivel requiere al menos 10 propiedades comparables. Se utiliza el primer nivel que alcanza dicho mínimo.

Cuando la propiedad evaluada pertenece al universo de referencia, se excluye a sí misma del cálculo. De esta manera, el requisito de 10 comparables representa efectivamente 10 propiedades distintas de la observación evaluada.

In [17]:
# 7.1 BENCHMARK JERÁRQUICO CON LEAVE-ONE-OUT

from src.benchmark import benchmark_jerarquico, NIVELES_COMPARABLES

MIN_COMPARABLES = 10

bm = benchmark_jerarquico(
    df, valor="USD_m2_Homogeneizado", niveles=NIVELES_COMPARABLES,
    mask_referencia=df["Flag_Comparable_Valido"],
    min_n=MIN_COMPARABLES, leave_one_out=True,
)

df["Benchmark_USD_m2"] = bm["Q50"]
df["P25_USD_m2_Comparable"] = bm["Q25"]
df["N_Comparables"] = bm["N"]
df["Nivel_Benchmark"] = bm["Nivel"]

con_bm = df["Benchmark_USD_m2"].notna()
print(f"Con benchmark: {con_bm.sum()} de {len(df)} ({con_bm.mean() * 100:.2f}%)")

print("\nNivel de la jerarquía utilizado:")
display(df["Nivel_Benchmark"].value_counts(dropna=False).sort_index().to_frame("Avisos"))

display(
    df.loc[con_bm, ["N_Comparables", "Benchmark_USD_m2", "P25_USD_m2_Comparable"]]
    .describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).round(1)
)
print("Benchmarks con menos de", MIN_COMPARABLES, "comparables:",
      int((df["N_Comparables"] < MIN_COMPARABLES).sum()))

Con benchmark: 9450 de 9499 (99.48%)

Nivel de la jerarquía utilizado:


,Avisos
Nivel_Benchmark,
1,4424
2,3396
3,1426
4,204
<NA>,49


,N_Comparables,Benchmark_USD_m2,P25_USD_m2_Comparable
count,9450.0,9450.0,9450.0
mean,25.2,2230.1,1958.7
std,15.8,589.1,509.6
min,10.0,760.2,672.0
1%,10.0,1053.9,935.5
25%,14.0,1807.8,1581.0
50%,21.0,2210.2,1923.3
75%,32.0,2567.8,2288.5
99%,86.0,3885.6,3304.4
max,133.0,6894.0,5901.9


Benchmarks con menos de 10 comparables: 0


**Resultado.** El 99,5% de los avisos (9.450) tiene un benchmark con al menos 10 comparables. El 46,8% se resolvió en el nivel más fino (barrio, ambientes, superficie y antigüedad), el 35,9% en el nivel 2, el 15,1% al ampliar a la comuna y solo el 2,2% en el nivel más amplio. La mediana es de 21 comparables por propiedad. Los 49 avisos sin benchmark son unidades de segmentos poco frecuentes en su comuna, que no se clasifican.

### 7.2 Gap de subvaluación y clasificación de oportunidades

A partir del benchmark se calcula el precio esperado de cada propiedad y su gap de subvaluación respecto del precio publicado:

- Precio esperado = Benchmark USD/m² × Superficie homogeneizada
- Gap de subvaluación (%) = (Precio esperado − Precio publicado) / Precio esperado × 100

Cada propiedad se clasifica según la definición operativa de propiedad barata (ver README):

- **Barata:** gap igual o superior al umbral (25%, recalibrado a partir del análisis de sensibilidad de la sección 7.3; el supuesto inicial era 15%).

- **Barata — requiere verificación:** cumple el criterio, pero tiene algún indicador de posible error de carga: valor atípicamente bajo en precio, superficie o precio por m², combinación inusual de variables, coordenadas inválidas o ficha sin descripción. Estos casos se revisan manualmente antes de incluirlos en el ranking.
- **A refaccionar con descuento:** necesita obra y tiene gap positivo. Se evalúa con el PER y no con el criterio de barata.
- **Precio de mercado:** no cumple el criterio.
- **Sin benchmark:** ningún nivel de la jerarquía alcanza el mínimo de comparables.

In [18]:
# 7.2 PRECIO ESPERADO, GAP Y CLASIFICACIÓN

from src.benchmark import clasificar_oportunidad

UMBRAL_GAP = 25   # % — recalibrado a partir de la sensibilidad (7.3); el supuesto inicial era 15%

df["Precio_Esperado"] = df["Benchmark_USD_m2"] * df["Superficie_Homogeneizada_m2"]
df["Gap_Subvaluacion_pct"] = (
    (df["Precio_Esperado"] - df["Precio_Valor"]) / df["Precio_Esperado"] * 100
)
df["Clasificacion_Oportunidad"] = clasificar_oportunidad(df, umbral_gap=UMBRAL_GAP)

print("Distribución del gap de subvaluación (%):")
display(
    df["Gap_Subvaluacion_pct"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).round(1).to_frame()
)

print("Clasificación de oportunidades:")
display(auditar_categorica(df, "Clasificacion_Oportunidad"))

print("Control — necesita obra clasificada como barata:",
      int((df["Estado_Propiedad"].eq("Necesita_obra")
           & df["Clasificacion_Oportunidad"].str.startswith("Barata")).sum()))

Distribución del gap de subvaluación (%):


,Gap_Subvaluacion_pct
count,9450.0
mean,-4.2
std,30.7
min,-274.1
10%,-36.7
25%,-15.9
50%,0.5
75%,13.8
90%,25.7
95%,32.9


Clasificación de oportunidades:


,Clasificacion_Oportunidad,Cantidad,Porcentaje
0,Precio_de_mercado,8252,86.87
1,Barata,716,7.54
2,A_refaccionar_con_descuento,362,3.81
3,Barata_requiere_verificacion,120,1.26
4,Sin_benchmark,49,0.52


Control — necesita obra clasificada como barata: 0


**Resultado.** El gap de subvaluación tiene una mediana de 0,5%, lo que indica que el benchmark está bien centrado: la propiedad típica se publica a un precio similar al de sus comparables. La dispersión es amplia (el 10% de los avisos supera un gap del 25,7% y otro 10% está más de 36% por encima de su benchmark), lo que refleja diferencias que las variables disponibles no capturan, como la orientación, el piso o la calidad de las terminaciones.

Con el umbral de 25%, 836 avisos (8,8%) cumplen el criterio de propiedad barata; 120 de ellos tienen indicadores de posible error de carga y requieren verificación antes del ranking. Además, 362 de las 439 unidades que necesitan obra (82,5%) se publican por debajo de su benchmark, lo que constituye una primera evidencia a favor de H1 (descuento por condición), que se analizará en el EDA.

### 7.3 Sensibilidad del umbral de gap

El umbral del 15% es un supuesto inicial. Para evaluar cuán sensible es la cantidad de oportunidades a ese supuesto, se repite la clasificación con umbrales del 10%, 15%, 20% y 25%. Un umbral adecuado debe identificar un conjunto de oportunidades acotado, que el fondo pueda revisar en la práctica, sin depender de un valor arbitrario.

In [19]:
# 7.3 SENSIBILIDAD DEL UMBRAL DE GAP

UMBRALES = [10, 15, 20, 25]

sensibilidad = pd.DataFrame({
    f"Umbral {u}%": clasificar_oportunidad(df, umbral_gap=u).value_counts()
    for u in UMBRALES
}).fillna(0).astype(int)

orden = ["Barata", "Barata_requiere_verificacion", "A_refaccionar_con_descuento",
         "Precio_de_mercado", "Sin_benchmark"]
sensibilidad = sensibilidad.reindex(orden)

display(sensibilidad)

print("Porcentaje de la base clasificada como barata (incluye las que requieren verificación):")
display(
    (sensibilidad.loc[["Barata", "Barata_requiere_verificacion"]].sum() / len(df) * 100)
    .round(1).rename("% de la base").to_frame()
)

,Umbral 10%,Umbral 15%,Umbral 20%,Umbral 25%
Clasificacion_Oportunidad,,,,
Barata,2092,1620,1103,716
Barata_requiere_verificacion,155,143,130,120
A_refaccionar_con_descuento,362,362,362,362
Precio_de_mercado,6841,7325,7855,8252
Sin_benchmark,49,49,49,49


Porcentaje de la base clasificada como barata (incluye las que requieren verificación):


,% de la base
Umbral 10%,23.7
Umbral 15%,18.6
Umbral 20%,13.0
Umbral 25%,8.8


**Resultado.** La cantidad de oportunidades es muy sensible al umbral: pasa del 23,7% de la base con un umbral del 10% al 8,8% con uno del 25%. Con el umbral inicial del 15%, casi uno de cada cinco avisos resultaba barato, lo que no constituye una señal de subvaluación sino la dispersión normal del mercado: el percentil 75 del gap ya es del 13,8%. Se adopta un umbral del **25%**, cercano al percentil 90 del gap, de modo que el criterio identifica aproximadamente al 10% de las propiedades con mayor descuento respecto de sus comparables. Cuando se valide el benchmark, se verificará además que el umbral supere su error típico de estimación.

## 8. Precio Esperado Reciclado (PER)

Para las propiedades clasificadas como `Necesita_obra` se estima el Precio Esperado Reciclado (PER), que representa su valor potencial luego de una refacción.

La estimación utiliza propiedades comparables clasificadas como `Buen_estado` o `Reciclada_refaccionada`. Se excluyen observaciones con indicadores de valores extremos, cantidad de ambientes dudosa y menciones correspondientes únicamente a reciclados parciales.

Se utiliza la misma jerarquía de comparables definida para el benchmark general, con un mínimo de 10 observaciones por grupo.

Además de la mediana como escenario base, se estiman los percentiles 25 y 75 para representar escenarios conservador y optimista del valor post-refacción.

In [20]:
# 8.1 PRECIO ESPERADO RECICLADO (PER)

mask_objetivo_per = df["Estado_Propiedad"].eq("Necesita_obra")

mask_ref_per = (
    df["Estado_Agrupado"].eq("Buen_o_reciclada")
    & ~df["Reciclado_Parcial"]
    & ~df["Flag_Extremo"].fillna(False).astype(bool)
    & ~df["Flag_Ambientes_Dudoso"].fillna(False).astype(bool)
)

per = benchmark_jerarquico(
    df, valor="USD_m2_Homogeneizado", niveles=NIVELES_COMPARABLES,
    mask_referencia=mask_ref_per, mask_objetivo=mask_objetivo_per,
    min_n=MIN_COMPARABLES, leave_one_out=False,
)

df["PER_USD_m2_P25"] = per["Q25"]
df["PER_USD_m2_Base"] = per["Q50"]
df["PER_USD_m2_P75"] = per["Q75"]
df["PER_N_Comparables"] = per["N"]
df["PER_Nivel"] = per["Nivel"]

escenarios = {"Conservador": "PER_USD_m2_P25", "Base": "PER_USD_m2_Base", "Optimista": "PER_USD_m2_P75"}
for escenario, columna in escenarios.items():
    df[f"PER_{escenario}"] = df[columna] * df["Superficie_Homogeneizada_m2"]

con_per = mask_objetivo_per & df["PER_Base"].notna()
print("Propiedades que necesitan obra:", int(mask_objetivo_per.sum()))
print("Comparables de referencia (buen estado o recicladas):", int(mask_ref_per.sum()))
print(f"Con PER: {con_per.sum()} ({con_per.sum() / mask_objetivo_per.sum() * 100:.1f}%)")

print("\nNivel de la jerarquía utilizado:")
display(df.loc[mask_objetivo_per, "PER_Nivel"].value_counts(dropna=False).sort_index().to_frame("Avisos"))

Propiedades que necesitan obra: 439
Comparables de referencia (buen estado o recicladas): 2844
Con PER: 428 (97.5%)

Nivel de la jerarquía utilizado:


,Avisos
PER_Nivel,
1,106
2,122
3,145
4,55
<NA>,11


**Resultado.** Se estimó el valor post-obra para 428 de las 439 propiedades que necesitan obra (97,5%), a partir de 2.844 comparables en buen estado o reciclados. Como el universo de referencia es más chico que el del benchmark general, una proporción mayor de las estimaciones se resolvió en los niveles 3 y 4 de la jerarquía (47%). Esos PER son menos precisos y se interpretan con más cautela.

### 8.2 Potencial bruto de revalorización

Para cada propiedad que necesita obra se calcula la diferencia porcentual entre su valor post-obra estimado y su precio publicado, en los tres escenarios:

Potencial bruto (%) = (PER − Precio publicado) / Precio publicado × 100

Este indicador **no es el margen de flipping**: no descuenta el costo de la obra, los costos de compra y venta ni el costo de tenencia. Indica cuánto se revaloriza la unidad según los comparables, y funciona como cota superior del margen. El margen potencial se calculará cuando estén definidos los parámetros de costo de la operación.

In [21]:
# 8.2 POTENCIAL BRUTO POR ESCENARIO

for escenario in escenarios:
    df[f"Potencial_Bruto_{escenario}_pct"] = (
        (df[f"PER_{escenario}"] - df["Precio_Valor"]) / df["Precio_Valor"] * 100
    )

columnas_potencial = [f"Potencial_Bruto_{e}_pct" for e in escenarios]
display(
    df.loc[con_per, columnas_potencial]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).round(1)
)

for escenario in escenarios:
    positivos = (df.loc[con_per, f"Potencial_Bruto_{escenario}_pct"] > 0).mean() * 100
    print(f"Potencial bruto positivo — escenario {escenario.lower()}: {positivos:.1f}% de las propiedades")

,Potencial_Bruto_Conservador_pct,Potencial_Bruto_Base_pct,Potencial_Bruto_Optimista_pct
count,428.0,428.0,428.0
mean,13.7,28.7,46.4
std,29.0,34.2,40.0
min,-69.2,-64.7,-53.1
10%,-16.1,-5.6,6.2
25%,-4.3,6.9,19.9
50%,10.2,23.1,39.6
75%,27.2,44.1,64.6
90%,47.2,65.9,90.0
max,184.4,227.5,305.3


Potencial bruto positivo — escenario conservador: 67.1% de las propiedades
Potencial bruto positivo — escenario base: 84.6% de las propiedades
Potencial bruto positivo — escenario optimista: 93.9% de las propiedades


**Resultado.** En la mediana, el valor post-obra estimado supera al precio publicado en un 10,2% en el escenario conservador, un 23,1% en el base y un 39,6% en el optimista. El 84,6% de las propiedades tiene un potencial bruto positivo en el escenario base, pero solo el 67,1% lo mantiene en el conservador.

Estos valores **no son márgenes de flipping**: todavía no descuentan el costo de la obra, los costos de compra y venta (que en conjunto pueden superar el 10% del precio) ni el costo de tenencia. Una parte importante de los potenciales moderados desaparecería al incorporar esos costos. El margen se calculará cuando se definan los parámetros de costo; por ahora, el potencial bruto permite descartar las propiedades sin margen posible, cuyo valor post-obra no supera el precio publicado.

## 9. Imputación de expensas

Las expensas se necesitan para estimar el costo de tenencia en el análisis de flipping. En la limpieza (paso 10 del notebook 02) se mostró que sus faltantes no son aleatorios: faltan con mayor frecuencia en las unidades más caras y más grandes. Por eso no se imputan con un valor central, sino con un método condicional (KNN), que estima cada faltante a partir de propiedades similares. La variable original se conserva sin cambios y la imputada se guarda en una columna separada, junto con un indicador.

Antes de imputar se revisan los valores informados más bajos. En la limpieza se reemplazaron por faltantes las expensas iguales a 0 (salvo las de avisos que aclaran que no pagan expensas), pero pueden quedar otros valores de relleno, como montos de un peso, que no representan expensas reales y distorsionarían la imputación.

In [22]:
# 9.1 DIAGNÓSTICO DE EXPENSAS

expensas = df["Expensas"]
print(f"Expensas faltantes: {expensas.isna().sum()} ({expensas.isna().mean() * 100:.1f}%)")

print("\nLos 15 valores informados más bajos (ARS):")
display(expensas.value_counts().sort_index().head(15).rename("Avisos").to_frame())

print("Avisos por rango de expensas (ARS):")
rangos = pd.cut(
    expensas,
    bins=[-0.1, 0, 1, 1_000, 10_000, 30_000, 50_000, np.inf],
    labels=["0", "1", "2 a 1.000", "1.001 a 10.000", "10.001 a 30.000",
            "30.001 a 50.000", "Más de 50.000"],
)
display(rangos.value_counts(sort=False).rename("Avisos").to_frame())

sin_expensas = df["Texto_Normalizado"].str.contains(PATRONES["sin_expensas"], regex=True)
print("Expensas = 0 en avisos que dicen 'sin expensas':",
      int((expensas.eq(0) & sin_expensas).sum()))
print("Expensas menores a 10.000 en avisos que dicen 'sin expensas':",
      int((expensas.between(0.5, 10_000) & sin_expensas).sum()))

Expensas faltantes: 982 (10.3%)

Los 15 valores informados más bajos (ARS):


,Avisos
Expensas,
0.0,43
1.0,93
4.0,2
10.0,3
11.0,1
60.0,2
70.0,1
87.0,1
90.0,1


Avisos por rango de expensas (ARS):


,Avisos
Expensas,
0,43
1,93
2 a 1.000,33
1.001 a 10.000,14
10.001 a 30.000,66
30.001 a 50.000,105
Más de 50.000,8163


Expensas = 0 en avisos que dicen 'sin expensas': 43
Expensas menores a 10.000 en avisos que dicen 'sin expensas': 0


**Resultado.** Además de los 982 faltantes (10,3%), hay 140 avisos con expensas informadas entre ARS 1 y ARS 10.000. Son valores implausibles como monto mensual: equivalen a menos de unos pocos dólares, y la mayoría son exactamente ARS 1, un valor típico de relleno. Algunos podrían estar cargados en otra escala (por ejemplo, en miles), pero no hay forma de verificarlo. Se reemplazan por faltantes antes de imputar. Las 43 expensas iguales a 0 se conservan, porque todos esos avisos aclaran que la unidad no paga expensas. Los valores de entre ARS 10.000 y 50.000 también se conservan: son bajos, pero pueden corresponder a edificios pequeños sin encargado ni servicios.

In [23]:
# 9.2 ¿EXPENSAS BAJAS DESACTUALIZADAS? RELACIÓN CON LA ANTIGÜEDAD DEL AVISO

# Los IDs de RE/MAX son crecientes: un ID más bajo indica una publicación más antigua
rangos_exp = pd.cut(
    df["Expensas"],
    bins=[-0.1, 0, 10_000, 30_000, 50_000, 100_000, np.inf],
    labels=["0 (sin expensas)", "1 a 10.000", "10.001 a 30.000", "30.001 a 50.000",
            "50.001 a 100.000", "Más de 100.000"],
)

print("ID del aviso según el rango de expensas (ID más bajo = publicación más antigua):")
display(
    df.groupby(rangos_exp, observed=True)["ID"]
    .describe(percentiles=[0.25, 0.5, 0.75])[["count", "25%", "50%", "75%"]]
    .round(0)
)

# Proporción de avisos en el cuartil de IDs más bajos, por rango de expensas
id_p25 = df["ID"].quantile(0.25)
print(f"\nProporción de avisos en el 25% más antiguo (ID < {id_p25:,.0f}):")
display(
    df.assign(Aviso_antiguo=df["ID"] < id_p25)
    .groupby(rangos_exp, observed=True)["Aviso_antiguo"].mean()
    .mul(100).round(1).rename("% avisos antiguos").to_frame()
)

ID del aviso según el rango de expensas (ID más bajo = publicación más antigua):


,count,25%,50%,75%
Expensas,,,,
0 (sin expensas),43.0,566730.0,618207.0,645090.0
1 a 10.000,140.0,550858.0,602886.0,635428.0
10.001 a 30.000,66.0,533096.0,594384.0,630204.0
30.001 a 50.000,105.0,559451.0,609306.0,631673.0
50.001 a 100.000,951.0,571700.0,611728.0,636359.0
Más de 100.000,7212.0,579882.0,614978.0,638950.0



Proporción de avisos en el 25% más antiguo (ID < 577,405):


,% avisos antiguos
Expensas,
0 (sin expensas),27.9
1 a 10.000,35.7
10.001 a 30.000,36.4
30.001 a 50.000,27.6
50.001 a 100.000,28.1
Más de 100.000,23.9


**Resultado.** Como la fuente no informa la fecha de publicación, se usa el identificador del aviso como aproximación, bajo el supuesto de que RE/MAX asigna los identificadores en forma creciente (el mismo supuesto adoptado en el tratamiento de duplicados). Las expensas de entre ARS 10.000 y 30.000 aparecen con mayor frecuencia en los avisos más antiguos: el 36% pertenece al cuartil de identificadores más bajos, contra entre 24% y 28% en el resto de los rangos. Esto sugiere que una parte de esos montos son expensas desactualizadas, publicadas hace tiempo y no actualizadas frente a la inflación.

In [24]:
# 9.3 EXPENSAS ALTAS: ¿CORRESPONDEN A UNIDADES GRANDES O DE LUJO?

df["Expensas_por_m2"] = df["Expensas"] / df["Superficie_Homogeneizada_m2"]

print("Expensas por m² (ARS):")
display(df["Expensas_por_m2"].describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_frame())

print("Los 10 avisos con mayor expensa por m²:")
display(
    df.nlargest(10, "Expensas_por_m2")[
        ["ID", "Titulo", "Barrio_Oficial", "Superficie_Homogeneizada_m2",
         "Precio_Valor", "Expensas", "Expensas_por_m2"]
    ]
)

Expensas por m² (ARS):


,Expensas_por_m2
count,8517.0
mean,3804.0
std,1918.0
min,0.0
50%,3585.0
90%,6174.0
99%,9549.0
max,24860.0


Los 10 avisos con mayor expensa por m²:


,ID,Titulo,Barrio_Oficial,Superficie_Homogeneizada_m2,Precio_Valor,Expensas,Expensas_por_m2
958,651328,Venta monoambiente en apart con cochera piso 11,Monserrat,32.180,54900.0,800000.0,24860.161591
2834,634365,Departamento en Hotel de Las Naciones al frente,San Nicolas,23.000,45000.0,490000.0,21304.347826
5440,605190,Monoambiente en Retiro con cochera,Retiro,27.500,63000.0,450000.0,16363.636364
7722,560837,Venta Monoambiente amoblado SLS Puerto Madero,Puerto Madero,49.500,300000.0,750000.0,15151.515152
1468,647040,PALACIO ALCORTA -en duplex con 2 suites y terraza,Palermo,180.500,785000.0,2659000.0,14731.301939
517,655044,"Venta Monoambiente con balcon al frente, PPatr...",Parque Patricios,21.030,40000.0,300000.0,14265.335235
511,655104,VENTA 3 AMB CON COCHERA BAULERA TORRE DE CATEG...,Palermo,63.115,330000.0,900000.0,14259.684703
6194,594617,Monoambiente Recoleta - Ideal Inversion Airbnb,Retiro,29.020,105000.0,412000.0,14197.105445
1565,646279,VENTA 3 AMBIENTES PALERMO PISO ALTO BALCON COC...,Palermo,44.305,145000.0,583219.0,13163.728699
2166,640761,Departamento 2 ambientes en venta en Las Cañitas,Palermo,46.500,190000.0,600000.0,12903.225806


**Resultado.** La expensa mediana es de ARS 3.585 por m², y el percentil 99, de ARS 9.549. Los valores más altos corresponden principalmente a unidades en apart-hoteles o condo-hoteles, donde las expensas incluyen servicios hoteleros, y a edificios de categoría con amenities completos. Son montos elevados pero coherentes con el tipo de edificio, por lo que se conservan.

In [25]:
# 9.4 VALORES DE RELLENO EN EXPENSAS

UMBRAL_EXPENSAS_MIN = 30_000   # ARS — por debajo (y por encima de 0): relleno o monto desactualizado

mask_relleno = df["Expensas"].gt(0) & df["Expensas"].lt(UMBRAL_EXPENSAS_MIN)

df = modificar(
    df, mask_relleno, "Expensas", np.nan,
    paso="Expensas: valores de relleno o desactualizados",
    criterio=f"Expensas entre ARS 1 y ARS {UMBRAL_EXPENSAS_MIN:,} → relleno o monto desactualizado",
    registro=registro,
)
df["Expensas_Informadas"] = df["Expensas"].notna()

print("Expensas pasadas a faltante:", int(mask_relleno.sum()))
print(f"Faltantes totales: {df['Expensas'].isna().sum()} ({df['Expensas'].isna().mean() * 100:.1f}%)")

Expensas pasadas a faltante: 189
Faltantes totales: 1171 (12.3%)


**Resultado.** Se reemplazaron por faltantes 189 expensas mayores a 0 y menores a ARS 30.000, por tratarse de valores de relleno (la mayoría, exactamente ARS 1) o de montos implausibles como gasto mensual en 2026, con indicios de estar desactualizados. Se conservan las 43 expensas iguales a 0, que corresponden a avisos sin expensas. En total, 1.171 avisos (12,3%) quedan sin expensas informadas y se imputan en el paso siguiente.

### 9.5 Selección del modelo de imputación

El modelo KNN estima las expensas de cada aviso a partir de las de sus vecinos más parecidos. Para elegir las variables y la cantidad de vecinos (k), se ocultan expensas conocidas y se mide qué tan bien las recupera el modelo, mediante validación cruzada de 5 particiones. Se comparan dos conjuntos de variables:

- **A:** superficie, ambientes, antigüedad, barrio y amenities del edificio.
- **B:** las mismas variables más el precio por m². El diagnóstico de faltantes de la limpieza mostró que la ausencia de expensas se asocia con el precio por m², por lo que esa variable puede mejorar la estimación. Como las expensas solo se usan para el costo de tenencia y no intervienen en el gap de subvaluación, incluirla no genera circularidad.

El modelo predice el logaritmo de las expensas, porque la variable es muy asimétrica; el error se informa en pesos.

In [26]:
# 9.5 SELECCIÓN DE VARIABLES Y DE K PARA KNN

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_predict

df["log_Superficie"] = np.log(df["Superficie_Homogeneizada_m2"])
df["log_USD_m2"] = np.log(df["USD_m2_Homogeneizado"])

# Amenities del edificio + indicador de si el aviso informa amenities
# (sin el indicador, "no informa" se confunde con "no tiene")
VARIABLES_EDIFICIO = ["Pileta", "SUM", "Gimnasio", "Laundry", "Seguridad",
                      "Solarium", "Ascensor", "Cochera_Incluida", "Amenities_Informadas"]
CONJUNTOS = {
    "A (sin precio)": ["log_Superficie", "Ambientes"],
    "B (con USD/m²)": ["log_Superficie", "Ambientes", "log_USD_m2"],
}
CATEGORICAS = ["Barrio_Oficial", "Banda_Antiguedad"]
VALORES_K = [15, 20, 30, 50]

def modelo_knn(numericas, k):
    preprocesador = ColumnTransformer([
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAS),
        ("edif", "passthrough", VARIABLES_EDIFICIO),
    ])
    return Pipeline([("prep", preprocesador),
                     ("knn", KNeighborsRegressor(n_neighbors=k, weights="distance", n_jobs=-1))])

obs = df.loc[df["Expensas"].gt(0)].copy()        # se excluyen los 0 ("sin expensas")
obs[VARIABLES_EDIFICIO] = obs[VARIABLES_EDIFICIO].astype(int)
y_log = np.log(obs["Expensas"])
cv = KFold(n_splits=5, shuffle=True, random_state=42)

resultados = []
for nombre, numericas in CONJUNTOS.items():
    X = obs[numericas + CATEGORICAS + VARIABLES_EDIFICIO]
    for k in VALORES_K:
        pred = np.exp(cross_val_predict(modelo_knn(numericas, k), X, y_log, cv=cv))
        error = (pred - obs["Expensas"]).abs()
        resultados.append({"Variables": nombre, "k": k,
                           "MAE (ARS)": round(error.mean()),
                           "Error absoluto mediano (ARS)": round(error.median()),
                           "MAE / mediana (%)": round(error.mean() / obs["Expensas"].median() * 100, 1)})

resultados = pd.DataFrame(resultados).sort_values("MAE (ARS)")
display(resultados)

mejor = resultados.iloc[0]
print(f"Mejor configuración: {mejor['Variables']}, k = {mejor['k']}")

,Variables,k,MAE (ARS),Error absoluto mediano (ARS),MAE / mediana (%)
5,B (con USD/m²),20,78219,50415,39.1
6,B (con USD/m²),30,78308,50000,39.2
4,B (con USD/m²),15,78615,50751,39.3
7,B (con USD/m²),50,79432,50089,39.7
2,A (sin precio),30,81038,51827,40.5
1,A (sin precio),20,81114,52359,40.6
3,A (sin precio),50,81598,51690,40.8
0,A (sin precio),15,81673,52690,40.8


Mejor configuración: B (con USD/m²), k = 20


**Resultado.** Incluir el precio por m² mejora la imputación en todas las cantidades de vecinos evaluadas. La mejor configuración (variables B, k = 20) tiene un error absoluto medio de ARS 78.219, equivalente al 39,1% de la mediana de las expensas, frente al 40,5% sin el precio. El error sigue siendo elevado porque las expensas dependen de características que la base no informa, como la existencia de encargado, la cantidad de unidades del edificio o los servicios incluidos. Por eso la imputación sirve para estimar el costo de tenencia en términos agregados, y no para conocer el monto exacto de cada aviso.

### 9.6 Imputación definitiva

Se entrena el modelo con la configuración de menor error (variables y cantidad de vecinos) sobre todas las expensas informadas, y se estiman las faltantes. La variable original `Expensas` no se modifica: el resultado se guarda en `Expensas_Imputadas`, y `Flag_Expensas_Imputadas` identifica los valores estimados. Las expensas iguales a 0 (avisos sin expensas) se mantienen en 0.

Como el diagnóstico de faltantes mostró que las expensas faltan con mayor frecuencia en las unidades más caras, se espera que los valores imputados sean, en promedio, más altos que los observados en los segmentos de mayor precio. Esto se verifica comparando ambos grupos por cuartil de precio por m².

In [27]:
# 9.6 IMPUTACIÓN DEFINITIVA DE EXPENSAS

numericas_mejor = CONJUNTOS[mejor["Variables"]]
k_mejor = int(mejor["k"])
columnas_x = numericas_mejor + CATEGORICAS + VARIABLES_EDIFICIO

modelo_final = modelo_knn(numericas_mejor, k_mejor)
modelo_final.fit(obs[columnas_x], y_log)

faltan = df["Expensas"].isna()
x_faltan = df.loc[faltan, columnas_x].copy()
x_faltan[VARIABLES_EDIFICIO] = x_faltan[VARIABLES_EDIFICIO].astype(int)

df["Expensas_Imputadas"] = df["Expensas"]
df.loc[faltan, "Expensas_Imputadas"] = np.exp(modelo_final.predict(x_faltan))
df["Flag_Expensas_Imputadas"] = faltan

print(f"Modelo: {mejor['Variables']}, k = {k_mejor}")
print("Expensas imputadas:", int(faltan.sum()))
print("Faltantes después de imputar:", int(df["Expensas_Imputadas"].isna().sum()))
print("Valores originales modificados:",
      int((df.loc[~faltan, "Expensas"] != df.loc[~faltan, "Expensas_Imputadas"]).sum()))

# Comparación por cuartil de precio por m²
cuartil_precio = pd.qcut(df["USD_m2_Homogeneizado"], 4,
                         labels=["Q1 (más baratos)", "Q2", "Q3", "Q4 (más caros)"])
comparacion = (
    df.assign(Cuartil_USD_m2=cuartil_precio,
              Tipo=np.where(faltan, "Imputadas", "Observadas"))
    .loc[lambda d: d["Expensas_Imputadas"].gt(0)]
    .groupby(["Cuartil_USD_m2", "Tipo"], observed=True)["Expensas_Imputadas"]
    .median().unstack().round(0)
)
print("\nMediana de expensas (ARS) por cuartil de USD/m²:")
display(comparacion)

print("\nAvisos que informan amenities:")
print(f"   entre las expensas observadas: {df.loc[~faltan, 'Amenities_Informadas'].mean() * 100:.1f}%")
print(f"   entre las expensas imputadas:  {df.loc[faltan, 'Amenities_Informadas'].mean() * 100:.1f}%")

Modelo: B (con USD/m²), k = 20
Expensas imputadas: 1171
Faltantes después de imputar: 0
Valores originales modificados: 0

Mediana de expensas (ARS) por cuartil de USD/m²:


Tipo,Imputadas,Observadas
Cuartil_USD_m2,,
Q1 (más baratos),144653.0,169210.0
Q2,169682.0,180000.0
Q3,179408.0,200000.0
Q4 (más caros),235501.0,275000.0



Avisos que informan amenities:
   entre las expensas observadas: 83.3%
   entre las expensas imputadas:  78.7%


**Resultado.** Se imputaron 1.171 expensas (12,3% de los avisos) sin modificar ningún valor original. Sin embargo, en todos los cuartiles de precio por m² la mediana de las expensas imputadas es menor que la de las observadas (entre 6% y 15%). La diferencia no se explica por la falta de información sobre amenities: la proporción de avisos con amenities informadas es similar en ambos grupos (79% y 83%). Para determinar si la diferencia proviene del método, se valida el modelo sobre expensas conocidas.

### 9.7 Validación de la imputación

Las expensas imputadas resultan, en la mediana, más bajas que las observadas en todos los cuartiles de precio por m². Para determinar si la diferencia se debe a los avisos imputados o al método, se aplica el mismo modelo a avisos con expensas conocidas, mediante validación cruzada (cada aviso se estima sin usar su propio valor), y se compara lo estimado con lo observado. Si el modelo también subestima los valores conocidos, la diferencia se debe al método: al promediar vecinos, el KNN atenúa los valores altos.

In [28]:
# 9.7 VALIDACIÓN DE LA IMPUTACIÓN: ¿EL MODELO SUBESTIMA?

pred_cv = np.exp(cross_val_predict(
    modelo_knn(numericas_mejor, k_mejor), obs[columnas_x], y_log, cv=cv
))

validacion = pd.DataFrame({
    "Cuartil_USD_m2": cuartil_precio.loc[obs.index],
    "Observada": obs["Expensas"],
    "Estimada": pred_cv,
})

tabla = validacion.groupby("Cuartil_USD_m2", observed=True)[["Observada", "Estimada"]].median().round(0)
tabla["Estimada / Observada"] = (tabla["Estimada"] / tabla["Observada"]).round(3)
print("Mediana de expensas en avisos con valor conocido (ARS):")
display(tabla)

FACTOR_AJUSTE = validacion["Observada"].sum() / validacion["Estimada"].sum()
print(f"Relación global estimada / observada (medianas): "
      f"{validacion['Estimada'].median() / validacion['Observada'].median():.3f}")
print(f"Suma estimada / suma observada: {1 / FACTOR_AJUSTE:.3f}")
print(f"Factor de ajuste para las imputaciones: {FACTOR_AJUSTE:.3f}")

Mediana de expensas en avisos con valor conocido (ARS):


,Observada,Estimada,Estimada / Observada
Cuartil_USD_m2,,,
Q1 (más baratos),169210.0,162444.0,0.960
Q2,180000.0,173532.0,0.964
Q3,200000.0,196106.0,0.981
Q4 (más caros),275000.0,252477.0,0.918


Relación global estimada / observada (medianas): 0.938
Suma estimada / suma observada: 0.894
Factor de ajuste para las imputaciones: 1.119


**Resultado.** Aplicado a expensas conocidas, el modelo subestima: la mediana estimada queda entre 2% y 8% por debajo de la observada según el cuartil, con la mayor diferencia en las unidades más caras, y la suma de los valores estimados representa el 89,4% de la suma observada. Es un efecto esperable del método: al promediar las expensas de los vecinos, el KNN atenúa los valores altos. Esta subestimación explica una parte de la diferencia observada en la sección 9.6; el resto puede deberse a que los avisos sin expensas tienen, dentro de cada cuartil, características asociadas a montos menores. Como el costo de tenencia depende del nivel promedio de las expensas, se corrige el nivel de los valores imputados con un factor de 1,119 (el cociente entre la suma observada y la estimada).

### 9.8 Ajuste de nivel de las expensas imputadas

La validación muestra que el modelo subestima las expensas conocidas: la suma de los valores estimados queda por debajo de la suma de los observados. Como las expensas imputadas se usarán para estimar el costo de tenencia, que depende de su nivel promedio, se aplica a los valores imputados un factor de ajuste igual al cociente entre la suma observada y la suma estimada en la validación. El ajuste corrige el nivel promedio, pero no la dispersión: las expensas imputadas siguen siendo menos variables que las reales, por lo que sirven para estimaciones agregadas y no para conocer el monto exacto de cada aviso.

In [29]:
# 9.8 AJUSTE DE NIVEL DE LAS EXPENSAS IMPUTADAS

df.loc[faltan, "Expensas_Imputadas"] = np.exp(modelo_final.predict(x_faltan)) * FACTOR_AJUSTE

comparacion_ajustada = (
    df.assign(Cuartil_USD_m2=cuartil_precio,
              Tipo=np.where(faltan, "Imputadas (ajustadas)", "Observadas"))
    .loc[lambda d: d["Expensas_Imputadas"].gt(0)]
    .groupby(["Cuartil_USD_m2", "Tipo"], observed=True)["Expensas_Imputadas"]
    .median().unstack().round(0)
)
print(f"Factor aplicado: {FACTOR_AJUSTE:.3f}")
print("Mediana de expensas (ARS) por cuartil de USD/m²:")
display(comparacion_ajustada)
print("Valores originales modificados:",
      int((df.loc[~faltan, "Expensas"] != df.loc[~faltan, "Expensas_Imputadas"]).sum()))

Factor aplicado: 1.119
Mediana de expensas (ARS) por cuartil de USD/m²:


Tipo,Imputadas (ajustadas),Observadas
Cuartil_USD_m2,,
Q1 (más baratos),161807.0,169210.0
Q2,189804.0,180000.0
Q3,200683.0,200000.0
Q4 (más caros),263428.0,275000.0


Valores originales modificados: 0


**Resultado.** Con el factor de ajuste, la mediana de las expensas imputadas queda a menos de 5% de la observada en todos los cuartiles de precio por m², y las diferencias que persisten van en ambos sentidos (por encima en el segundo cuartil, por debajo en el primero y el cuarto), sin un sesgo sistemático. Los valores originales no se modificaron. `Expensas_Imputadas` queda disponible para estimar el costo de tenencia, con la precaución de que representa un nivel esperado y no el monto exacto de cada unidad. Las expensas están expresadas en pesos; su conversión a dólares se hará al definir la fuente del tipo de cambio.

## 10. Muestra para la validación manual del estado

Para estimar la precisión de la clasificación de `Estado_Propiedad`, se selecciona una muestra aleatoria estratificada de 50 avisos por categoría. Cada aviso se revisa manualmente a partir de su título y descripción, y la clasificación manual se compara con la automática. El objetivo es una precisión de al menos 80% en `Necesita_obra` y `Reciclada_refaccionada`, que son las categorías que intervienen en el benchmark y en el Precio Esperado Reciclado. Si una categoría no alcanza ese nivel, se usará solo su señal fuerte.

La muestra se guarda en `reports/validacion_estado_muestra.csv`, con una columna `Estado_Manual` vacía para completar durante la revisión.

In [30]:
# 10. MUESTRA PARA LA VALIDACIÓN MANUAL DEL ESTADO

N_VALIDACION = 50
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)
RUTA_VALIDACION = REPORTS_DIR / "validacion_estado_muestra.csv"

if RUTA_VALIDACION.exists():
    # La muestra ya está en el repositorio: no se sobrescribe, porque puede
    # contener la revisión manual (columna Estado_Manual)
    muestra_estado = pd.read_csv(RUTA_VALIDACION)
    print("La muestra ya existe en el repositorio y no se sobrescribe.")
else:
    muestra_estado = pd.concat([
        grupo.sample(min(N_VALIDACION, len(grupo)), random_state=42)
        for _, grupo in df.groupby("Estado_Propiedad")
    ])[["ID", "Estado_Propiedad", "Reciclado_Parcial", "Titulo", "Descripcion"]].assign(Estado_Manual="")
    muestra_estado.to_csv(RUTA_VALIDACION, index=False)
    print(f"Muestra guardada en {RUTA_VALIDACION}")

print("Avisos por categoría en la muestra:")
display(muestra_estado["Estado_Propiedad"].value_counts().to_frame("Avisos"))

La muestra ya existe en el repositorio y no se sobrescribe.
Avisos por categoría en la muestra:


,Avisos
Estado_Propiedad,
Buen_estado,50
Necesita_obra,50
Reciclada_refaccionada,50
Sin_clasificar,50


**Resultado.** Se seleccionaron 50 avisos de cada categoría de `Estado_Propiedad` (200 en total) y se exportaron en `reports/validacion_estado_muestra.csv`. La revisión manual de esta muestra permitirá estimar la precisión de la clasificación automática. Sus resultados se incorporarán en la siguiente etapa.

## 11. Validación del benchmark y del Precio Esperado Reciclado

Para evaluar la confiabilidad de ambos estimadores se aplica un esquema de holdout: se separa al azar el 20% de las propiedades de referencia como conjunto de prueba (estratificando por barrio), se calcula su benchmark usando únicamente el 80% restante y se compara la estimación con su precio por m² real. El procedimiento se repite con 5 semillas distintas y se informan la media y el desvío de cada métrica:

- **MAPE:** error porcentual absoluto medio.
- **MdAPE:** error porcentual absoluto mediano, más robusto frente a valores extremos.
- **Cobertura del rango intercuartílico:** proporción de precios reales comprendidos entre el percentil 25 y el 75 del grupo de comparables. Un valor cercano al 50% indica que el rango representa bien la dispersión del segmento.

La validación del benchmark usa como universo los comparables válidos, y la del PER, las propiedades en buen estado o recicladas. Como cada estimación se calcula con el 80% de las referencias, los grupos son algo más chicos que en el cálculo definitivo, por lo que los errores obtenidos son una estimación ligeramente pesimista.

El criterio de aceptación inicial es un MAPE menor al 15%. Además, el umbral de gap adoptado para la definición de propiedad barata (25%) debe superar el error típico del benchmark, para que una propiedad clasificada como barata no lo sea solo por error de estimación.

In [31]:
# 11. VALIDACIÓN DEL BENCHMARK Y DEL PER (HOLDOUT 80/20)

from sklearn.model_selection import train_test_split

SEMILLAS = [0, 1, 2, 3, 4]

def validar_estimador(mask_universo, nombre):
    universo = df.index[mask_universo]
    # Estratificación por barrio (los barrios con muy pocos casos se agrupan)
    barrio = df.loc[universo, "Barrio_Oficial"]
    estratos = barrio.where(barrio.map(barrio.value_counts()) >= 5, "Otros")
    # Si algún estrato queda con menos de 2 casos, se suma al más frecuente
    conteo = estratos.value_counts()
    estratos = estratos.replace(dict.fromkeys(conteo[conteo < 2].index, conteo.idxmax()))

    filas = []
    for semilla in SEMILLAS:
        train, test = train_test_split(universo, test_size=0.2,
                                       random_state=semilla, stratify=estratos)
        referencia = pd.Series(df.index.isin(train), index=df.index)
        objetivo = pd.Series(df.index.isin(test), index=df.index)
        est = benchmark_jerarquico(df, "USD_m2_Homogeneizado", NIVELES_COMPARABLES,
                                   referencia, objetivo, min_n=MIN_COMPARABLES,
                                   leave_one_out=False).loc[test]
        real = df.loc[test, "USD_m2_Homogeneizado"]
        ok = est["Q50"].notna()
        error_pct = ((est.loc[ok, "Q50"] - real[ok]).abs() / real[ok]) * 100
        filas.append({
            "Cobertura (%)": ok.mean() * 100,
            "MAPE (%)": error_pct.mean(),
            "MdAPE (%)": error_pct.median(),
            "Dentro de P25–P75 (%)": real[ok].between(est.loc[ok, "Q25"], est.loc[ok, "Q75"]).mean() * 100,
        })
    resumen = pd.DataFrame(filas).agg(["mean", "std"]).round(1)
    resumen.index = [f"{nombre} — media", f"{nombre} — desvío"]
    return resumen

validacion_benchmark = pd.concat([
    validar_estimador(df["Flag_Comparable_Valido"], "Benchmark"),
    validar_estimador(mask_ref_per, "PER"),
])
display(validacion_benchmark)

,Cobertura (%),MAPE (%),MdAPE (%),Dentro de P25–P75 (%)
Benchmark — media,99.6,19.5,14.7,43.6
Benchmark — desvío,0.1,0.5,0.4,1.3
PER — media,96.3,17.6,14.0,45.1
PER — desvío,0.7,0.5,0.7,1.1


**Resultado.** El benchmark tiene un error porcentual absoluto mediano de 14,7% y un error medio (MAPE) de 19,5%. El PER muestra valores similares: 14,0% y 17,6%. Los resultados son estables entre particiones, con desvíos menores a un punto porcentual.

El MAPE supera el criterio de aceptación inicial del 15%, porque está influido por propiedades cuyo precio difiere mucho del de sus comparables. El error típico, medido por la mediana, se ubica cerca de ese umbral. Esto indica que las variables disponibles explican una parte importante del precio por m², pero no toda: características como la orientación, el piso o la calidad de las terminaciones no se observan y generan diferencias de alrededor del 15% respecto del benchmark.

Esta validación respalda el umbral de gap adoptado en la sección 7.3. Un umbral del 25% supera tanto el error típico como el error medio del benchmark, de modo que una propiedad clasificada como barata no lo es solo por imprecisión de la estimación. En cambio, con el umbral inicial del 15%, una parte de las oportunidades identificadas habría quedado dentro del margen de error del método.

Por último, el rango entre los percentiles 25 y 75 del grupo contiene al 44% de los precios reales, algo menos del 50% esperado. La dispersión real es levemente mayor que la que reflejan los grupos de comparables, por lo que la condición del cuartil inferior debe interpretarse con esa cautela.

## 12. Cierre y exportación

Se exporta la base con las variables derivadas y los KPIs, que es el insumo del análisis exploratorio. Antes de exportar se eliminan las columnas auxiliares usadas solo durante el procesamiento (los textos unificados y normalizados, que duplican el título y la descripción, y las transformaciones logarítmicas usadas en la imputación). También se exportan el registro de los pasos de este notebook y el resumen de la validación del benchmark.

| Archivo | Contenido |
| --- | --- |
| `data/processed/remax_deptos_features.csv` | Base analítica con variables derivadas y KPIs |
| `data/processed/registro_features.csv` | Registro de exclusiones y modificaciones de este notebook |
| `reports/validacion_benchmark.csv` | Métricas de la validación del benchmark y del PER |

In [32]:
# 12.1 EXPORTACIÓN

COLUMNAS_AUXILIARES = ["Texto_Analisis", "Texto_Normalizado", "log_Superficie", "log_USD_m2"]

df_final = df.drop(columns=[c for c in COLUMNAS_AUXILIARES if c in df.columns])

RUTA_FEATURES = PROCESSED_DIR / "remax_deptos_features.csv"
RUTA_REGISTRO = PROCESSED_DIR / "registro_features.csv"
RUTA_VALIDACION_BM = REPORTS_DIR / "validacion_benchmark.csv"

df_final.to_csv(RUTA_FEATURES, index=False)
tabla_registro(registro).to_csv(RUTA_REGISTRO, index=False)
validacion_benchmark.to_csv(RUTA_VALIDACION_BM)

print("Dimensiones de la base final:", df_final.shape)
print("IDs duplicados:", int(df_final["ID"].duplicated().sum()))
print(f"Exportado: {RUTA_FEATURES} ({RUTA_FEATURES.stat().st_size / 1e6:.1f} MB)")
print(f"Exportado: {RUTA_REGISTRO}")
print(f"Exportado: {RUTA_VALIDACION_BM}")

print("\nColumnas de la base final:")
print(list(df_final.columns))

Dimensiones de la base final: (9499, 99)
IDs duplicados: 0
Exportado: data/processed/remax_deptos_features.csv (24.9 MB)
Exportado: data/processed/registro_features.csv
Exportado: reports/validacion_benchmark.csv

Columnas de la base final:
['ID', 'ID_Entidad', 'Operacion', 'Tipo_Propiedad', 'Titulo', 'Descripcion', 'Fecha_Publicacion', 'Antiguedad', 'Apto_Credito', 'Amenities', 'Moneda', 'Precio_Valor', 'Moneda_Expensas', 'Expensas', 'Precio_m2_Total', 'Precio_m2_Cubierto', 'Ambientes', 'Dormitorios', 'Baños', 'Superficie_Total_m2', 'Superficie_Cubierta_m2', 'Direccion', 'Barrio', 'Ubicacion', 'Latitud', 'Longitud', 'Estado_Publicacion', 'ID_Interno', 'Link', 'Tipo_Propiedad_Original', 'Ambientes_Titulo', 'Flag_Ambientes_Dudoso', 'Flag_Dormitorios_Inconsistente', 'Flag_Posible_Republicacion', 'Expensas_Informadas', 'Antiguedad_Informada', 'Amenities_Informadas', 'Flag_Atipico_Bajo', 'Flag_Atipico_Alto', 'Flag_Extremo', 'Flag_Outlier_Multivariado', 'Barrio_Oficial', 'Comuna', 'Flag_Coo

**Resultado.** Se exportó la base con 9.499 avisos y 99 variables: las 41 de la base limpia más las variables derivadas del texto, el estado de la propiedad, la ubicación oficial, la superficie homogeneizada, los KPIs de subvaluación y del Precio Esperado Reciclado, y las expensas imputadas. No hay identificadores duplicados.

### 12.2 Diccionario de datos

Se genera el diccionario de datos de la base final, con el significado, el tipo, la unidad, la fuente y la transformación aplicada a cada variable, además de su porcentaje de faltantes. Las descripciones están en `data/diccionario_variables.csv`; el tipo de dato y los faltantes se calculan directamente sobre la base. El resultado se guarda en `data/diccionario_datos.md`.

In [33]:
# 12.2 DICCIONARIO DE DATOS

descripciones = pd.read_csv("data/diccionario_variables.csv")

sin_descripcion = sorted(set(df_final.columns) - set(descripciones["Variable"]))
print("Variables sin descripción:", sin_descripcion or "ninguna")

diccionario = (
    descripciones.set_index("Variable")
    .reindex(df_final.columns)
    .assign(Tipo=df_final.dtypes.astype(str),
            Faltantes=(df_final.isna().mean() * 100).round(1))
    .rename_axis("Variable").reset_index()
    [["Variable", "Descripcion", "Tipo", "Unidad", "Fuente", "Transformacion", "Faltantes"]]
    .rename(columns={"Descripcion": "Descripción", "Transformacion": "Transformación",
                     "Faltantes": "Faltantes (%)"})
)

# Tabla en formato Markdown
lineas = ["| " + " | ".join(diccionario.columns) + " |",
          "|" + " --- |" * len(diccionario.columns)]
lineas += ["| " + " | ".join(str(v) for v in fila) + " |"
           for fila in diccionario.fillna("").values]

encabezado = (
    "# Diccionario de datos\n\n"
    "Base: `data/processed/remax_deptos_features.csv` "
    f"({len(df_final):,} avisos, {df_final.shape[1]} variables). "
    "Fecha de corte de la extracción: 26/09/2026 (aproximada).\n\n"
)
RUTA_DICCIONARIO = Path("data/diccionario_datos.md")
RUTA_DICCIONARIO.write_text(encabezado + "\n".join(lineas) + "\n", encoding="utf-8")

print(f"Diccionario guardado en {RUTA_DICCIONARIO} ({len(diccionario)} variables)")
display(diccionario.head(10))

Variables sin descripción: ninguna
Diccionario guardado en data/diccionario_datos.md (99 variables)


,Variable,Descripción,Tipo,Unidad,Fuente,Transformación,Faltantes (%)
0,ID,Identificador del aviso en RE/MAX,int64,—,RE/MAX (base raw),Sin cambios,0.0
1,ID_Entidad,Identificador interno de la entidad en la API,object,—,RE/MAX (base raw),Sin cambios,0.0
2,Operacion,Tipo de operación (constante: venta),object,—,RE/MAX (base raw),Sin cambios,0.0
3,Tipo_Propiedad,Tipología homogeneizada (constante: departamento),object,—,RE/MAX (base raw),Subcategorías de departamento unificadas; la o...,0.0
4,Titulo,Título del aviso,object,Texto,RE/MAX (base raw),Sin cambios,0.0
5,Descripcion,Descripción del aviso,object,Texto,RE/MAX (base raw),Sin cambios; falta en 54 avisos por falla del ...,0.6
6,Fecha_Publicacion,Fecha de publicación (no informada por la fuente),float64,Fecha,RE/MAX (base raw),100% faltante; se conserva para documentar la ...,100.0
7,Antiguedad,Antigüedad del edificio,float64,Años,RE/MAX (base raw),Valores 0 en stock usado reemplazados por falt...,3.4
8,Apto_Credito,Indica si el aviso declara apto crédito hipote...,object,Booleano,RE/MAX (base raw),Sin cambios,0.6
9,Amenities,Lista de amenities informada por la fuente,object,Texto,RE/MAX (base raw),Sin cambios,17.3


**Resultado.** El diccionario documenta las 99 variables de la base final. Para cada una indica su significado, tipo de dato, unidad, fuente (base raw, limpieza, ingeniería de variables o BA Data), transformación aplicada y porcentaje de faltantes.

## 13. Síntesis

**Variables construidas.** A partir de la base limpia de 9.500 avisos se construyó una base analítica de 9.499 departamentos usados y 99 variables. Se excluyó un aviso ubicado fuera de CABA.

- **Ubicación:** cada aviso se asignó a su barrio y comuna oficiales por unión espacial con los límites de BA Data. La etiqueta de la fuente coincide con el barrio oficial solo en el 83,8% de los casos, en parte por subzonas informales y en parte por la tendencia a etiquetar avisos con el barrio vecino de mayor valor.
- **Texto:** se extrajeron características de los avisos (luminosidad, vista, orientación, apto profesional, reciclado, necesidad de refacción) y amenities, con control de negación y combinando la lista estructurada de la fuente con el texto.
- **Estado:** el 63,5% de los avisos no informa el estado del inmueble; el 4,6% (439) necesita obra y el 31,9% está en buen estado o reciclado.

**KPIs.**

- **Benchmark de comparables:** cubre al 99,5% de los avisos, con una mediana de 21 comparables por propiedad. Su error típico de estimación es de 14,7% (MdAPE).
- **Propiedad barata:** con un umbral de gap del 25%, recalibrado a partir del análisis de sensibilidad, 836 avisos (8,8%) cumplen el criterio, de los cuales 120 requieren verificación manual por indicadores de posible error de carga.
- **Flipping:** 362 de las 439 unidades que necesitan obra (82,5%) se publican por debajo de su benchmark, una primera evidencia a favor de H1. El valor post-obra estimado supera al precio publicado en una mediana del 23,1% en el escenario base, antes de descontar costos.
- **Expensas:** se imputaron 1.171 valores faltantes o implausibles mediante KNN, con un ajuste de nivel validado sobre expensas conocidas.

**Limitaciones.**

- `Estado_Propiedad` es una señal imperfecta. Su validación manual está preparada (`reports/validacion_estado_muestra.csv`), pero pendiente.
- El benchmark no supera el criterio de aceptación inicial en términos de error medio (MAPE de 19,5% frente a 15%): las variables disponibles no capturan características como la orientación, el piso o la calidad de las terminaciones.
- Todos los precios son de publicación y no de cierre, por lo que el gap es un descuento relativo a la oferta.
- La antigüedad de los avisos se aproxima por su identificador, bajo el supuesto de que se asignan en forma creciente.
- Las expensas imputadas representan un nivel esperado y no el monto exacto de cada unidad; están expresadas en pesos.

**Próximos pasos.**

- Completar la validación manual del estado.
- Definir los costos de la operación (obra, transacción, tenencia y contingencias) para pasar del potencial bruto al margen de flipping.
- Evaluar la sensibilidad del coeficiente de superficie descubierta.
- Avanzar con el análisis exploratorio (notebook 04) y los tests preliminares de H1 y H2.